![LangChain](../rag/img/langchain.jpeg)

# Deep Agents

Les **Deep Agents** sont une bibliothèque standalone construite au-dessus de **LangChain** et **LangGraph**.
Ils fournissent une architecture prête à l'emploi pour créer des agents capables de :

- **Planifier** des tâches complexes et les décomposer automatiquement,
- **Déléguer** le travail à des **subagents** spécialisés avec **isolation de contexte**,
- Gérer un **système de fichiers virtuel** pour la mémoire longue durée,
- Charger des **Skills** (compétences) à la demande pour économiser les tokens,
- Intégrer un flux **Human-in-the-Loop** pour valider les actions sensibles.

Contrairement à un simple agent LangGraph, un Deep Agent embarque nativement la planification,
la délégation et la gestion de contexte — sans configuration manuelle du graphe.

> 💡 **Prérequis** : avoir suivi `rag.ipynb` et compris les concepts de base de LangGraph (agent, mémoire, thread_id).

# 1. Installation et configuration
___

La bibliothèque `deepagents` s'installe via pip. Elle inclut automatiquement LangChain et LangGraph comme dépendances.

In [ ]:
!pip install -qU deepagents langchain-mistralai

In [ ]:
import os
from datetime import datetime

from IPython.display import display, Markdown
from dotenv import load_dotenv
from langchain.tools import tool
from deepagents import create_deep_agent
from deep_agents.checkpoint.memory import MemorySaver

load_dotenv(override=True)

# 2. Premier Deep Agent
___

La fonction `create_deep_agent` est le point d'entrée principal.
Elle crée un agent complet avec planification et gestion de contexte intégrées.

On lui fournit :
- un **modèle** LLM (ici Mistral via API, ou Ollama en local),
- des **outils** (`@tool`) que l'agent peut appeler,
- un **system prompt** optionnel pour définir son rôle.

In [ ]:
@tool
def get_current_weather(city: str) -> str:
    """Retourne la météo actuelle pour une ville donnée."""
    # Simulation — en production, appeler une vraie API météo
    weather_data = {
        "Paris": "15°C, nuageux",
        "Lyon": "18°C, ensoleillé",
        "Marseille": "22°C, ensoleillé",
    }
    return weather_data.get(city, f"Météo non disponible pour {city}")


@tool
def get_current_time() -> str:
    """Retourne la date et l'heure actuelles."""
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")


# Création de l'agent
agent = create_deep_agent(
    model="mistral:mistral-large-latest",
    # model="ollama:llama3",  # Alternative locale (nécessite Ollama)
    tools=[get_current_weather, get_current_time],
    system_prompt=(
        "Tu es un assistant francophone. "
        "Utilise les outils disponibles pour répondre aux questions."
    ),
)

# Invocation
result = agent.invoke(
    {"messages": [{"role": "user", "content": "Quelle heure est-il et quel temps fait-il à Paris ?"}]}
)

display(Markdown(result["messages"][-1].content))

# 3. Subagents
___

### Pourquoi des subagents ?

Quand une tâche devient complexe, un seul agent atteint ses limites :
- La fenêtre de contexte se remplit de détails non pertinents,
- L'agent perd le fil entre recherche, rédaction, analyse...

Les **subagents** résolvent ce problème grâce à **l'isolation de contexte** :
chaque subagent reçoit uniquement la description de sa tâche et ses outils spécifiques.

L'agent principal agit comme un **chef de projet** : il planifie, délègue, et synthétise.

### Architecture parent → subagents

```
┌─────────────────────────┐
│     Agent Principal       │
│  (planification + délégation) │
└────────────┬────────────┘
             │
     ┌───────┼───────┐
     │               │
     ▼               ▼
┌──────────┐  ┌──────────┐
│ Recherche  │  │ Rédaction  │
│ Subagent   │  │ Subagent   │
└──────────┘  └──────────┘
  🔍 outils       ✍️ outils
  spécifiques     spécifiques
```

Chaque subagent est défini comme un dictionnaire avec :
- `name` : identifiant unique,
- `description` : quand l'agent principal doit déléguer à ce subagent,
- `system_prompt` : instructions spécifiques,
- `tools` : outils exclusifs au subagent,
- `model` (optionnel) : override du modèle pour ce subagent.

In [ ]:
@tool
def web_search(query: str) -> str:
    """Recherche d'informations sur le web."""
    # Simulation — en production, utiliser Tavily ou SerpAPI
    return (
        f"Résultats pour '{query}' : "
        "LangGraph est un framework d'orchestration pour agents LLM, "
        "permettant de construire des applications stateful avec mémoire persistante."
    )


# Définition du subagent de recherche
research_subagent = {
    "name": "research-agent",
    "description": "Recherche approfondie sur des sujets techniques ou factuels.",
    "system_prompt": (
        "Tu es un chercheur spécialisé. Utilise l'outil de recherche web "
        "pour trouver des informations précises et retourne un résumé structuré."
    ),
    "tools": [web_search],
}

# Agent principal avec subagent
agent_with_sub = create_deep_agent(
    model="mistral:mistral-large-latest",
    # model="ollama:llama3",
    subagents=[research_subagent],
    system_prompt="Tu es un assistant qui délègue les recherches à ton subagent spécialisé.",
)

result = agent_with_sub.invoke(
    {"messages": [{"role": "user", "content": "Explique-moi ce qu'est LangGraph."}]}
)

display(Markdown(result["messages"][-1].content))

### Orchestration multi-subagents

On peut combiner plusieurs subagents. L'agent principal décide automatiquement
lequel mobiliser en fonction de la tâche demandée.

In [ ]:
@tool
def write_document(title: str, content: str) -> str:
    """Rédige un document structuré avec un titre et un contenu."""
    return f"# {title}\n\n{content}"


# Subagent rédacteur
writer_subagent = {
    "name": "writer-agent",
    "description": "Rédaction de documents, articles et synthèses.",
    "system_prompt": (
        "Tu es un rédacteur technique. Écris des textes clairs, "
        "structurés et adaptés au public cible."
    ),
    "tools": [write_document],
}

# Agent principal avec deux subagents
multi_agent = create_deep_agent(
    model="mistral:mistral-large-latest",
    # model="ollama:llama3",
    subagents=[research_subagent, writer_subagent],
    system_prompt=(
        "Tu es un chef de projet. Délègue les recherches au subagent de recherche "
        "et la rédaction au subagent rédacteur. Coordonne le tout pour produire un résultat final."
    ),
)

result = multi_agent.invoke(
    {"messages": [{
        "role": "user",
        "content": "Fais une recherche sur les Deep Agents et rédige un court article de synthèse."
    }]}
)

display(Markdown(result["messages"][-1].content))

# 4. Human-in-the-Loop
___

Certaines actions sont **sensibles** (supprimer un fichier, envoyer un email...).
Le mécanisme **Human-in-the-Loop** (HITL) permet d'**interrompre** l'agent
avant l'exécution d'un outil critique, pour que l'humain puisse :

- **Approuver** (`approve`) : l'outil s'exécute tel quel,
- **Modifier** (`edit`) : l'humain ajuste les paramètres avant exécution,
- **Rejeter** (`reject`) : l'outil n'est pas exécuté.

Pour activer le HITL, deux éléments sont nécessaires :
1. Un **checkpointer** (`MemorySaver`) pour persister l'état entre interruption et reprise,
2. Le paramètre `interrupt_on` qui définit quels outils déclenchent une interruption.

In [ ]:
import uuid
from deep_agents.types import Command


@tool
def delete_file(path: str) -> str:
    """Supprime un fichier du système de fichiers."""
    return f"Fichier supprimé : {path}"


@tool
def read_file(path: str) -> str:
    """Lit le contenu d'un fichier."""
    return f"Contenu du fichier {path} : [exemple de contenu]"


# Checkpointer obligatoire pour HITL
checkpointer = MemorySaver()

hitl_agent = create_deep_agent(
    model="mistral:mistral-large-latest",
    # model="ollama:llama3",
    tools=[delete_file, read_file],
    interrupt_on={
        "delete_file": True,   # Interruption avec approve/edit/reject
        "read_file": False,    # Pas d'interruption
    },
    checkpointer=checkpointer,
    system_prompt="Tu es un assistant de gestion de fichiers.",
)

# Créer un thread_id unique pour cette session
config = {"configurable": {"thread_id": str(uuid.uuid4())}}

# L'agent va tenter d'appeler delete_file → interruption
result = hitl_agent.invoke(
    {"messages": [{"role": "user", "content": "Supprime le fichier temp.txt"}]},
    config=config,
)

# Vérifier si l'exécution a été interrompue
if result.get("__interrupt__"):
    interrupts = result["__interrupt__"][0].value
    action_requests = interrupts["action_requests"]

    for action in action_requests:
        display(Markdown(
            f"**Interruption** : l'agent veut appeler `{action['name']}`\n\n"
            f"**Arguments** : `{action['args']}`"
        ))

    # Approuver l'action
    decisions = [{"type": "approve"}]

    result = hitl_agent.invoke(
        Command(resume={"decisions": decisions}),
        config=config,  # Même config pour reprendre le thread
    )

display(Markdown(result["messages"][-1].content))

# 5. Skills
___

Les **Skills** (compétences) sont un système de **progressive disclosure** :
au lieu de charger toutes les instructions dans le prompt système dès le départ,
l'agent découvre et charge les skills **à la demande**, quand c'est pertinent.

### Les 3 niveaux de chargement

| Niveau | Contenu | Quand |
|--------|---------|-------|
| 1. Métadonnées | `name` + `description` (~100 mots) | Toujours en contexte |
| 2. Corps SKILL.md | Instructions complètes (<5k mots) | Quand le skill est déclenché |
| 3. Ressources | Scripts, références, assets | Chargés à la demande par l'agent |

### Structure d'un skill

```
skills/
  summarize/
    SKILL.md          # Fichier requis (frontmatter YAML + instructions markdown)
    references/       # Documentation optionnelle
    scripts/          # Scripts exécutables optionnels
```

Le fichier `SKILL.md` contient un **frontmatter YAML** (name + description) suivi d'instructions markdown.

In [ ]:
from deepagents.backends import FilesystemBackend

# On utilise le dossier skills/ créé à côté de ce notebook
# Il contient : skills/summarize/SKILL.md
skills_dir = os.path.join(os.getcwd(), "skills")

skills_agent = create_deep_agent(
    model="mistral:mistral-large-latest",
    # model="ollama:llama3",
    skills=[skills_dir],
    backend=FilesystemBackend(root_dir=os.getcwd()),
    system_prompt="Tu es un assistant polyvalent. Utilise tes skills quand c'est pertinent.",
)

result = skills_agent.invoke(
    {"messages": [{
        "role": "user",
        "content": (
            "Résume le texte suivant : LangGraph est un framework d'orchestration "
            "pour construire des applications stateful avec des agents LLM. "
            "Il gère la mémoire, le routage et la persistance automatiquement."
        ),
    }]}
)

display(Markdown(result["messages"][-1].content))

# 6. Utilisation du CLI
___

La bibliothèque fournit également un **CLI** (`deepagents-cli`) pour interagir avec
un Deep Agent directement depuis le terminal, sans écrire de code Python.

### Installation

```bash
uv tool install deepagents-cli
```

### Commandes principales

```bash
# Démarrer une conversation
deepagents

# Reprendre une conversation précédente
deepagents --resume

# Charger des skills personnalisés
deepagents --skills ./skills/

# Utiliser un modèle spécifique
deepagents --model ollama:llama3
```

### Fonctionnalités du CLI

| Fonctionnalité | Description |
|----------------|-------------|
| Conversation | Chat interactif avec l'agent |
| Resume | Reprendre une session précédente |
| Skills | Chargement de skills personnalisés |
| Human-in-the-Loop | Approbation interactive des actions sensibles |
| Mémoire persistante | Historique conservé entre les sessions |

> ⚠️ Le CLI n'est pas exécutable directement dans un notebook Jupyter.
> Les exemples ci-dessus sont à lancer dans un terminal.

### Exercices

**Exercice 1** : Créez un Deep Agent avec **deux subagents** formant un pipeline de traduction :
1. Un subagent `translator` qui traduit un texte du français vers l'anglais,
2. Un subagent `reviewer` qui relit et corrige la traduction.
3. L'agent principal coordonne : il envoie le texte au traducteur, puis le résultat au relecteur.

**Exercice 2** : Ajoutez un mécanisme **Human-in-the-Loop** pour que l'utilisateur
puisse approuver ou rejeter la traduction finale avant qu'elle ne soit retournée.

**Bonus** : Créez un skill personnalisé `translation/SKILL.md` qui définit
des règles de style de traduction (registre formel, glossaire technique, etc.).

In [ ]:
# Votre code ici